# الفصل التاسع عشر: مسار تطوير الويب - بناء واجهة API لإدارة المهام (FastAPI)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-19.ipynb)

> **الوقت المتوقع:** ~9 دقيقة قراءة · **15** قسمًا · **18** مثالًا برمجيًا.

**كيف تستخدم هذا الدفتر؟**

1. شغّل الخلايا بالترتيب من الأعلى إلى الأسفل (`Shift` + `Enter`).
2. خلايا الشيفرة قابلة للتعديل: غيّر القيم وأعد التشغيل لتجرّب.
3. خلايا 🧾 معروضة للقراءة فقط (أوامر طرفية، أو إدخال يدوي، أو مقتطف ناقص)، وخلايا ⚠️ تُظهر خطأً مقصودًا يُلتقط تلقائيًا فلا يتوقف «تشغيل الكل».

> **Kaggle:** نزّل الملف من المستودع ثم `File ▸ Import Notebook`، أو استورده مباشرة من GitHub بعد ربط حسابك.

<details>
<summary><b>محتويات الفصل</b></summary>

- **19.1** لماذا يستحق هذا الفصل اهتمامك الكامل؟
- **19.2** المفاهيم الأساسية: كيف يعمل الويب؟
- **19.3** لماذا FastAPI؟
- **19.4** أول واجهة API: «مرحبًا أيها العالم»
- **19.5** المعاملات: Path Parameters و Query Parameters
- **19.6** Pydantic: نمذجة البيانات الواردة (Request Body)
- **19.7** معالجة الأخطاء: `HTTPException`
- **19.8** دعم async بشكل كامل
- **19.9** تنظيم مشروع أكبر: `APIRouter`
- **19.10** مشروع تطبيقي متكامل: واجهة API لإدارة المهام
- **19.11** أفضل الممارسات
- **19.12** تمارين
- **19.13** اختبر فهمك (Quiz)
- **19.14** ملخص الفصل (Cheat Sheet)
- **19.15** ماذا بعد؟

</details>

In [ ]:
#@title ⚙️ تهيئة البيئة  { display-mode: "form" }
import sys, os, random, pathlib

print("Python:", sys.version.split()[0], "|", sys.platform)
assert sys.version_info >= (3, 10), "هذا الكتاب يفترض Python 3.10 أو أحدث"

SEED = 42
random.seed(SEED)

# تثبيت الحزم الناقصة فقط (سريع إن كانت مثبّتة أصلًا)
import importlib.util as _iu
_need = [m for m in ["fastapi", "httpx", "pydantic"] if _iu.find_spec(m) is None]
if _need:
    %pip install -q fastapi httpx pydantic

> ℹ️ نستخدم `TestClient` بدل تشغيل خادم حقيقي، فيعمل الدفتر بلا منافذ.

## 19.1 لماذا يستحق هذا الفصل اهتمامك الكامل؟

كل تطبيق تستخدمه يوميًا تقريبًا — Instagram، تطبيق البنك، منصة التوصيل، حتى الطقس على هاتفك — يعتمد على نفس الفكرة الجوهرية: تطبيق (العميل/Client) يطلب بيانات أو يُرسلها إلى خادم بعيد (Server) عبر الإنترنت، والخادم يستجيب. هذه «اللغة» التي يتحدث بها التطبيق مع الخادم تُسمى **واجهة برمجة التطبيقات (API - Application Programming Interface)**.

حتى الآن، كل برامجنا كانت «منعزلة» — تعمل على جهازك فقط. في هذا الفصل، ستبني **أول خادم حقيقي بك**: برنامج يعمل بشكل دائم، منتظرًا طلبات من أي مكان في العالم (أو من جهازك فقط أثناء التعلم)، يستقبلها، يعالجها، ويرد عليها — تمامًا كما تفعل خوادم الشركات الكبرى، لكن بمقياس أصغر وتعليمي.

سنستخدم **FastAPI**: إطار عمل (Framework) حديث لبناء واجهات API بلغة Python، أصبح في السنوات الأخيرة من أكثر الأطر شعبية وانتشارًا عالميًا، بسبب سرعته العالية، وبساطته، واعتماده المباشر والعميق على **كل** ما تعلمناه في الفصول السابقة: Type Hints (الفصل 15)، async/await (الفصل 17)، وDataclasses/Pydantic لنمذجة البيانات (امتداد لمفاهيم الفصل 14). هذا الفصل هو **تتويج عملي حقيقي** لكل رحلتك في هذا الكتاب حتى الآن.

> 💡 **نصيحة الخبير:** مسار تطوير الويب بـ Python (باستخدام FastAPI أو Django) هو أحد أكثر المسارات طلبًا في سوق العمل البرمجي عالميًا. إتقان هذا الفصل يفتح لك بابًا حقيقيًا نحو وظائف "Backend Developer" أو "Full-Stack Developer".

## 19.2 المفاهيم الأساسية: كيف يعمل الويب؟

قبل كتابة أي كود، يجب فهم الصورة الكبرى.

### نموذج العميل-الخادم (Client-Server Model)

```
┌──────────┐        طلب (Request)         ┌──────────┐
│  العميل   │  ───────────────────────────>  │  الخادم   │
│ (Client) │                                │ (Server) │
│          │  <───────────────────────────  │          │
└──────────┘        استجابة (Response)       └──────────┘
```

- **العميل (Client)**: متصفحك، تطبيق الجوال، أو حتى برنامج Python آخر — أي طرف **يطلب** شيئًا.
- **الخادم (Server)**: برنامج يعمل بشكل دائم، **ينتظر** الطلبات ويستجيب لها. هذا ما سنبنيه اليوم!

### بروتوكول HTTP: لغة التواصل

**HTTP** (HyperText Transfer Protocol) هو «اللغة» المتفق عليها عالميًا لتبادل الطلبات والاستجابات عبر الويب. كل طلب HTTP يتكون من:

| العنصر | المعنى | مثال |
|---|---|---|
| **Method** (الفعل) | ماذا أريد أن أفعل؟ | `GET`, `POST`, `PUT`, `DELETE` |
| **URL** (المسار) | أين؟ | `/tasks/5` |
| **Headers** (ترويسات) | معلومات إضافية | نوع المحتوى، التوثيق |
| **Body** (الجسم) | البيانات المُرسَلة (إن وُجدت) | `{"title": "مهمة جديدة"}` |

### أفعال HTTP الأساسية (HTTP Methods)

| الفعل | الغرض | مثال عملي |
|---|---|---|
| `GET` | **قراءة/جلب** بيانات | جلب قائمة كل المهام |
| `POST` | **إنشاء** مورد جديد | إضافة مهمة جديدة |
| `PUT`/`PATCH` | **تحديث** مورد موجود | تعديل عنوان مهمة |
| `DELETE` | **حذف** مورد | حذف مهمة |

هذا النمط (ربط كل فعل HTTP بعملية CRUD: Create, Read, Update, Delete) يُسمى **REST** (Representational State Transfer) — وهو المعيار الأكثر انتشارًا عالميًا لتصميم واجهات API.

### رموز الحالة (HTTP Status Codes)

عندما يستجيب الخادم، يُرفق **رمز حالة** يُلخّص ماذا حدث:

| الفئة | المعنى | أمثلة شائعة |
|---|---|---|
| `2xx` | نجاح | `200 OK`, `201 Created`, `204 No Content` |
| `4xx` | خطأ من طرف العميل | `400 Bad Request`, `404 Not Found`, `422 Unprocessable Entity` |
| `5xx` | خطأ من طرف الخادم | `500 Internal Server Error` |

> 💡 **نصيحة الخبير:** إن رأيت `404` أثناء تصفحك العادي للإنترنت، فأنت الآن تعرف بالضبط ماذا تعني: «الخادم فهم طلبك، لكن المورد الذي طلبته غير موجود».

## 19.3 لماذا FastAPI؟

### 🔄 مقارنة: أطر عمل الويب الشائعة في Python

| الإطار | الطابع | الاستخدام الأمثل |
|---|---|---|
| **Django** | شامل جدًا (Batteries-Included) — يتضمن قاعدة بيانات، إدارة، مصادقة... | مواقع كبيرة متكاملة (كمواقع تجارة إلكترونية كاملة) |
| **Flask** | مرن وبسيط جدًا (Microframework) | مشاريع صغيرة إلى متوسطة، تحكم كامل بكل التفاصيل |
| **FastAPI** | حديث، سريع جدًا، يعتمد على Type Hints، توليد توثيق تلقائي | **واجهات API** حديثة (الأكثر ملاءمة لهذا الفصل) |

### لماذا FastAPI تحديدًا لتعلّم بناء APIs؟

1. **يعتمد مباشرة على Type Hints** التي أتقناها في الفصل 15 — لا مفاهيم جديدة غريبة، بل استخدام عملي لما تعرفه.
2. **تحقق تلقائي من صحة البيانات** (Validation) — إن أرسل العميل بيانات خاطئة، يرفضها FastAPI تلقائيًا برسالة خطأ واضحة، دون أن تكتب سطر تحقق واحد يدويًا.
3. **توثيق تفاعلي تلقائي** — بمجرد تشغيل الخادم، تحصل على واجهة ويب تفاعلية (Swagger UI) تعرض كل نقاط النهاية (Endpoints) ويمكنك تجربتها مباشرة من المتصفح.
4. **أداء عالٍ جدًا** — مبني على `async`/`await` (الفصل 17) من الأساس.

## 19.4 أول واجهة API: «مرحبًا أيها العالم»

### التثبيت

```bash
uv init task_api
cd task_api
uv add fastapi
uv add --dev "uvicorn[standard]"
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

- **FastAPI**: الإطار نفسه، مسؤول عن تعريف نقاط النهاية ومعالجة الطلبات.
- **Uvicorn**: خادم فعلي (ASGI Server) يُشغّل تطبيق FastAPI ويستقبل الطلبات الحقيقية عبر الشبكة. FastAPI «يصف» كيف يُعالَج كل طلب، وUvicorn هو من «يستمع فعليًا» على منفذ الشبكة وينفّذ هذا الوصف.

### الكود الأول

In [ ]:
# main.py
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def read_root() -> dict[str, str]:
    return {"message": "مرحبًا بك في واجهة API!"}

### تشغيل الخادم

```bash
uv run uvicorn main:app --reload
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

- `main:app`: يعني "من ملف `main.py`، استخدم المتغير `app`".
- `--reload`: يُعيد تشغيل الخادم تلقائيًا كلما عدّلت الكود أثناء التطوير — مفيد جدًا، لكن **لا تستخدمها في الإنتاج الفعلي**.

افتح متصفحك على `http://127.0.0.1:8000`، ستجد:

```json
{"message": "مرحبًا بك في واجهة API!"}
```

🧾 **للقراءة فقط** — محتوى ملف `json`.

### التوثيق التفاعلي التلقائي

الآن افتح `http://127.0.0.1:8000/docs` — ستُفاجأ بواجهة تفاعلية كاملة (Swagger UI) تعرض كل نقاط النهاية، تلقائيًا، بدون أن تكتب أي سطر توثيق إضافي! يمكنك تجربة كل Endpoint مباشرة من هذه الصفحة.

> 💡 **نصيحة الخبير:** هذه الميزة وحدها (التوثيق التلقائي من `/docs`) توفر ساعات عمل كانت تُصرف تقليديًا على كتابة توثيق منفصل يصبح قديمًا بسرعة. FastAPI يُولّد هذا التوثيق مباشرة من الكود نفسه (من الـ Type Hints والتعليقات)، فيبقى دائمًا **متزامنًا ودقيقًا 100%** مع الكود الفعلي.

## 19.5 المعاملات: Path Parameters و Query Parameters

### معاملات المسار (Path Parameters)

In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/tasks/{task_id}")
def get_task(task_id: int) -> dict[str, int]:
    return {"task_id": task_id}

جرّب `http://127.0.0.1:8000/tasks/5`:

```json
{"task_id": 5}
```

🧾 **للقراءة فقط** — محتوى ملف `json`.

لاحظ السحر: كتبنا `task_id: int`، فإن حاولت زيارة `/tasks/abc` (نص بدل رقم)، سيرفض FastAPI الطلب تلقائيًا:

```json
{
  "detail": [
    {
      "type": "int_parsing",
      "loc": ["path", "task_id"],
      "msg": "Input should be a valid integer"
    }
  ]
}
```

🧾 **للقراءة فقط** — محتوى ملف `json`.

**هذا هو جوهر الفائدة العملية لـ Type Hints**: لم نكتب أي كود تحقق يدوي، لكن FastAPI يتحقق تلقائيًا من النوع، ويرفض الطلبات غير الصحيحة برسالة واضحة، قبل أن يصل الطلب حتى إلى جسم دالتنا.

### معاملات الاستعلام (Query Parameters)

أي معامل في توقيع الدالة **لا يظهر** في مسار الـ URL يُعتبر تلقائيًا «معامل استعلام» (Query Parameter):

In [ ]:
@app.get("/tasks")
def list_tasks(skip: int = 0, limit: int = 10) -> dict[str, int]:
    return {"skip": skip, "limit": limit}

جرّب: `http://127.0.0.1:8000/tasks?skip=5&limit=20`

```json
{"skip": 5, "limit": 20}
```

🧾 **للقراءة فقط** — محتوى ملف `json`.

وإن لم تُحدد القيم: `http://127.0.0.1:8000/tasks` تستخدم القيم الافتراضية `skip=0, limit=10`.

### معاملات استعلام اختيارية

In [ ]:
@app.get("/search")
def search_tasks(keyword: str | None = None) -> dict[str, str | None]:
    if keyword is None:
        return {"message": "لم يُحدد كلمة بحث"}
    return {"message": f"البحث عن: {keyword}"}

> 🔄 **مقارنة: Path مقابل Query Parameters**
> - **Path**: جزء من «هوية» المورد نفسه (`/tasks/5` → المهمة رقم 5 تحديدًا).
> - **Query**: خيارات إضافية لتصفية أو تعديل الطلب (`/tasks?status=done` → كل المهام المنجزة).

## 19.6 Pydantic: نمذجة البيانات الواردة (Request Body)

حتى الآن تعاملنا مع بيانات بسيطة في المسار أو الاستعلام. لكن عند **إنشاء** مورد جديد (مثل إضافة مهمة)، نحتاج إرسال بيانات **مُهيكَلة أكثر** (عنوان، أولوية، إلخ) في «جسم الطلب» (Request Body). هنا تأتي **Pydantic** — المكتبة التي تعمل في قلب FastAPI.

### تعريف نموذج بيانات (Model)

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class TaskCreate(BaseModel):
    title: str
    priority: str = "متوسطة"
    done: bool = False

@app.post("/tasks")
def create_task(task: TaskCreate) -> TaskCreate:
    return task

لاحظ كيف تُشبه `TaskCreate` كتابة `@dataclass` من الفصل 14 — لكنها **أقوى بكثير**: `BaseModel` (من Pydantic) تُضيف **تحقق تلقائي صارم من صحة البيانات (Validation)** عند استقبال أي طلب.

جرّب إرسال طلب `POST` (عبر `/docs` أو أداة مثل `curl`) بجسم:

```json
{"title": "تعلم FastAPI"}
```

🧾 **للقراءة فقط** — محتوى ملف `json`.

النتيجة:

```json
{"title": "تعلم FastAPI", "priority": "متوسطة", "done": false}
```

🧾 **للقراءة فقط** — محتوى ملف `json`.

والآن جرّب إرسال بيانات خاطئة:

```json
{"priority": "عالية"}
```

🧾 **للقراءة فقط** — محتوى ملف `json`.

(بدون `title` الإلزامي):

```json
{
  "detail": [
    {
      "type": "missing",
      "loc": ["body", "title"],
      "msg": "Field required"
    }
  ]
}
```

🧾 **للقراءة فقط** — محتوى ملف `json`.

> 💡 **نصيحة الخبير:** هذا التحقق التلقائي من صحة البيانات هو **الفائدة الأعظم** لاستخدام Pydantic/FastAPI. في أطر عمل أقدم، كان على المبرمج كتابة عشرات الأسطر يدويًا للتحقق من كل حقل (هل موجود؟ هل من النوع الصحيح؟ هل ضمن النطاق المسموح؟). هنا، التحقق **مُشتق تلقائيًا** من Type Hints التي كتبتها أصلًا لتوثيق الكود.

### 🔄 مقارنة: `@dataclass` (الفصل 14) مقابل Pydantic `BaseModel`

| المعيار | `@dataclass` | Pydantic `BaseModel` |
|---|---|---|
| التحقق من الأنواع وقت التشغيل | لا (فقط عبر mypy، ساكن) | **نعم**، فعليًا عند الإنشاء |
| تحويل تلقائي من/إلى JSON | يدوي (`asdict`) | مدمج بالكامل (`.model_dump()`, `.model_validate_json()`) |
| رسائل خطأ واضحة عند بيانات خاطئة | لا، يحدث خطأ عادي لاحقًا | نعم، فورية ومفصلة |
| الاستخدام الأمثل | نماذج بيانات داخلية في برنامجك | بيانات قادمة من **مصدر خارجي غير موثوق** (كطلبات الويب) |

### قيود تحقق إضافية مع `Field`

In [ ]:
from pydantic import BaseModel, Field

class TaskCreate(BaseModel):
    title: str = Field(min_length=1, max_length=100)
    priority: str = Field(default="متوسطة", pattern="^(منخفضة|متوسطة|عالية)$")
    estimated_hours: float = Field(default=1.0, gt=0, le=100)

الآن إن أرسل العميل `title` فارغًا، أو `priority` بقيمة غير مدعومة، أو `estimated_hours` سالبة، يرفض FastAPI الطلب تلقائيًا برسالة دقيقة توضح بالضبط ما هو الخطأ.

### نماذج الاستجابة (Response Models)

يمكننا أيضًا تحديد **شكل البيانات المُرجَعة** صراحة، لضمان عدم تسريب حقول داخلية حساسة بالخطأ:

In [ ]:
class TaskOut(BaseModel):
    id: int
    title: str
    priority: str
    done: bool

@app.post("/tasks", response_model=TaskOut)
def create_task(task: TaskCreate) -> TaskOut:
    new_id = 1  # لاحقًا سيأتي من قاعدة بيانات حقيقية
    return TaskOut(id=new_id, title=task.title, priority=task.priority, done=task.done)

## 19.7 معالجة الأخطاء: `HTTPException`

عندما يطلب العميل موردًا غير موجود، يجب أن نُرجع رمز حالة مناسبًا (`404`) ورسالة واضحة، بدلًا من انهيار الخادم بخطأ غامض.

In [ ]:
from fastapi import FastAPI, HTTPException

app = FastAPI()

tasks_db: dict[int, str] = {1: "تعلم Python", 2: "تعلم FastAPI"}

@app.get("/tasks/{task_id}")
def get_task(task_id: int) -> dict[str, str]:
    if task_id not in tasks_db:
        raise HTTPException(status_code=404, detail="المهمة غير موجودة")
    return {"title": tasks_db[task_id]}

جرّب `/tasks/999`:

```json
{"detail": "المهمة غير موجودة"}
```

🧾 **للقراءة فقط** — محتوى ملف `json`.

برمز حالة `404` فعليًا (يمكن التحقق منه في تبويب Network بأدوات المطور في المتصفح، أو عبر `/docs`).

> 💡 **نصيحة الخبير:** `raise HTTPException(...)` تعمل تمامًا كأي استثناء آخر تعلمناه في الفصل 11 — FastAPI «يلتقط» هذا النوع المحدد من الاستثناءات تلقائيًا في الخلفية، ويحوّله إلى استجابة HTTP مناسبة بالرمز والرسالة التي حددتها.

## 19.8 دعم async بشكل كامل

كل نقطة نهاية (Endpoint) يمكن تعريفها كـ Coroutine باستخدام `async def`، مستفيدة من كل ما تعلمناه في الفصل 17، خاصة عند التعامل مع قواعد بيانات أو طلبات خارجية بطيئة:

In [ ]:
import asyncio

@app.get("/slow-operation")
async def slow_operation() -> dict[str, str]:
    await asyncio.sleep(2)  # محاكاة عملية بطيئة (استدعاء API خارجي، قاعدة بيانات، إلخ)
    return {"message": "اكتملت العملية البطيئة"}

> ⚠️ **تذكير من الفصل 17:** لا تستخدم `time.sleep()` أبدًا داخل دالة `async def`! هذا يُجمّد الخادم بأكمله عن خدمة أي طلب آخر أثناء الانتظار. استخدم دائمًا النسخ غير المتزامنة (`asyncio.sleep`, `httpx.AsyncClient`, إلخ).

> 💡 **نصيحة الخبير:** إن لم تكن نقطة النهاية تحتاج فعليًا لانتظار عملية I/O (كحساب بسيط أو قراءة من قائمة في الذاكرة)، يمكنك تعريفها كدالة عادية (`def` بدون `async`). FastAPI يتعامل بذكاء مع كلا النوعين بكفاءة.

## 19.9 تنظيم مشروع أكبر: `APIRouter`

عندما يكبر مشروعك، وضع كل نقاط النهاية في ملف `main.py` واحد يصبح غير عملي. نستخدم `APIRouter` لتقسيم الكود منطقيًا:

In [ ]:
# src/task_api/routers/tasks.py
from fastapi import APIRouter

router = APIRouter(prefix="/tasks", tags=["Tasks"])

@router.get("/")
def list_tasks() -> list[str]:
    return ["مهمة 1", "مهمة 2"]

```python
# src/task_api/main.py
from fastapi import FastAPI
from task_api.routers import tasks

app = FastAPI()
app.include_router(tasks.router)
```

🧾 **للقراءة فقط** — الوحدة `task_api` تُنشأ داخل مشروع محلي أو ليست حزمة عامة، فلا يمكن استيرادها هنا.

هذا يُشبه تمامًا فكرة تقسيم الكود إلى ملفات منفصلة (Modules) التي تعلمناها عند هيكلة المشاريع في الفصل 18 — `APIRouter` هو ببساطة «فئة فرعية» منظّمة من نقاط النهاية، تُدمج لاحقًا في التطبيق الرئيسي.

## 19.10 مشروع تطبيقي متكامل: واجهة API لإدارة المهام

لنبني الآن المشروع الكامل: واجهة API حقيقية لإدارة المهام، تدعم كل عمليات CRUD، مع حفظ دائم بصيغة JSON (معتمدين على ما تعلمناه في الفصل 16)، هيكلة احترافية (الفصل 18)، وType Hints كاملة (الفصل 15).

### هيكلية المشروع

```
task_api/
├── .venv/
├── pyproject.toml
├── src/
│   └── task_api/
│       ├── __init__.py
│       ├── main.py
│       ├── models.py
│       ├── storage.py
│       └── routers/
│           ├── __init__.py
│           └── tasks.py
└── tasks.json
```

<details>
<summary>✅ الحل الكامل</summary>

### `src/task_api/models.py`

```python
"""نماذج البيانات (Pydantic Models)."""
from __future__ import annotations

from typing import Literal
from pydantic import BaseModel, Field

type Priority = Literal["منخفضة", "متوسطة", "عالية"]

class TaskCreate(BaseModel):
    """البيانات المطلوبة لإنشاء مهمة جديدة."""
    title: str = Field(min_length=1, max_length=200)
    priority: Priority = "متوسطة"

class TaskUpdate(BaseModel):
    """البيانات القابلة للتحديث في مهمة موجودة (كلها اختيارية)."""
    title: str | None = Field(default=None, min_length=1, max_length=200)
    priority: Priority | None = None
    done: bool | None = None

class Task(BaseModel):
    """نموذج المهمة الكامل كما يُخزَّن ويُرجَع للعميل."""
    id: int
    title: str
    priority: Priority
    done: bool = False
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

### `src/task_api/storage.py`

```python
"""طبقة تخزين بسيطة تعتمد على ملف JSON (كما تعلمنا في الفصل 16)."""
from __future__ import annotations

import json
from pathlib import Path

from task_api.models import Task

class TaskStorage:
    def __init__(self, file_path: Path) -> None:
        self.file_path = file_path
        self._tasks: dict[int, Task] = {}
        self._next_id: int = 1
        self._load()

    def _load(self) -> None:
        if not self.file_path.exists():
            return
        try:
            raw_data = json.loads(self.file_path.read_text(encoding="utf-8"))
            self._tasks = {item["id"]: Task(**item) for item in raw_data}
            if self._tasks:
                self._next_id = max(self._tasks.keys()) + 1
        except (json.JSONDecodeError, KeyError):
            self._tasks = {}

    def _save(self) -> None:
        data = [task.model_dump() for task in self._tasks.values()]
        self.file_path.write_text(
            json.dumps(data, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    def get_all(self) -> list[Task]:
        return list(self._tasks.values())

    def get_by_id(self, task_id: int) -> Task | None:
        return self._tasks.get(task_id)

    def create(self, title: str, priority: str) -> Task:
        task = Task(id=self._next_id, title=title, priority=priority, done=False)  # type: ignore[arg-type]
        self._tasks[task.id] = task
        self._next_id += 1
        self._save()
        return task

    def update(self, task_id: int, **changes: object) -> Task | None:
        task = self._tasks.get(task_id)
        if task is None:
            return None
        updated_task = task.model_copy(update={k: v for k, v in changes.items() if v is not None})
        self._tasks[task_id] = updated_task
        self._save()
        return updated_task

    def delete(self, task_id: int) -> bool:
        if task_id not in self._tasks:
            return False
        del self._tasks[task_id]
        self._save()
        return True

# نسخة واحدة مشتركة للتطبيق بالكامل (Singleton بسيط)
storage = TaskStorage(Path("tasks.json"))
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

### `src/task_api/routers/tasks.py`

```python
"""نقاط النهاية الخاصة بإدارة المهام."""
from fastapi import APIRouter, HTTPException, status

from task_api.models import Task, TaskCreate, TaskUpdate
from task_api.storage import storage

router = APIRouter(prefix="/tasks", tags=["Tasks"])

@router.get("/", response_model=list[Task])
def list_tasks(done: bool | None = None) -> list[Task]:
    """يُرجع كل المهام، مع إمكانية التصفية حسب حالة الإنجاز."""
    all_tasks = storage.get_all()
    if done is None:
        return all_tasks
    return [task for task in all_tasks if task.done == done]

@router.get("/{task_id}", response_model=Task)
def get_task(task_id: int) -> Task:
    """يُرجع مهمة واحدة بمعرّفها."""
    task = storage.get_by_id(task_id)
    if task is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail=f"لا توجد مهمة بالمعرّف {task_id}",
        )
    return task

@router.post("/", response_model=Task, status_code=status.HTTP_201_CREATED)
def create_task(task_data: TaskCreate) -> Task:
    """ينشئ مهمة جديدة."""
    return storage.create(title=task_data.title, priority=task_data.priority)

@router.patch("/{task_id}", response_model=Task)
def update_task(task_id: int, task_data: TaskUpdate) -> Task:
    """يُحدّث حقولًا معيّنة في مهمة موجودة."""
    updated = storage.update(
        task_id,
        title=task_data.title,
        priority=task_data.priority,
        done=task_data.done,
    )
    if updated is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail=f"لا توجد مهمة بالمعرّف {task_id}",
        )
    return updated

@router.delete("/{task_id}", status_code=status.HTTP_204_NO_CONTENT)
def delete_task(task_id: int) -> None:
    """يحذف مهمة موجودة."""
    deleted = storage.delete(task_id)
    if not deleted:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail=f"لا توجد مهمة بالمعرّف {task_id}",
        )
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

### `src/task_api/main.py`

```python
"""نقطة الدخول الرئيسية لتطبيق FastAPI."""
from fastapi import FastAPI

from task_api.routers import tasks

app = FastAPI(
    title="واجهة API لإدارة المهام",
    description="مشروع تطبيقي من كتاب تعلّم Python",
    version="1.0.0",
)

app.include_router(tasks.router)

@app.get("/", tags=["Root"])
def read_root() -> dict[str, str]:
    return {"message": "مرحبًا بك في واجهة API لإدارة المهام! زُر /docs للتوثيق التفاعلي"}
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

### `pyproject.toml`

```toml
[project]
name = "task-api"
version = "1.0.0"
requires-python = ">=3.14"
dependencies = [
    "fastapi>=0.110.0",
    "pydantic>=2.6.0",
]

[dependency-groups]
dev = [
    "uvicorn[standard]>=0.29.0",
    "mypy>=1.8.0",
]

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"
```

🧾 **للقراءة فقط** — محتوى ملف `toml`.

### تشغيل المشروع

```bash
uv run uvicorn task_api.main:app --reload
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### تجربة الـ API

```bash
# إنشاء مهمة جديدة
curl -X POST http://127.0.0.1:8000/tasks/ \
  -H "Content-Type: application/json" \
  -d '{"title": "تعلم بناء APIs", "priority": "عالية"}'

# عرض كل المهام
curl http://127.0.0.1:8000/tasks/

# تحديث مهمة (وضعها كمنجزة)
curl -X PATCH http://127.0.0.1:8000/tasks/1 \
  -H "Content-Type: application/json" \
  -d '{"done": true}'

# حذف مهمة
curl -X DELETE http://127.0.0.1:8000/tasks/1
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

أو ببساطة أكبر: افتح `http://127.0.0.1:8000/docs` وجرّب كل العمليات بصريًا!

</details>

## 19.11 أفضل الممارسات

### افصل دائمًا بين النماذج (Models) والمنطق (Logic) والمسارات (Routers)

كما في المشروع أعلاه: `models.py` للبيانات، `storage.py` للمنطق، `routers/` للمسارات — لا تضع كل شيء في `main.py` واحد ضخم.

### استخدم `response_model` دائمًا

يضمن عدم تسريب بيانات داخلية حساسة بالخطأ، ويُحسّن التوثيق التلقائي في `/docs`.

### استخدم رموز حالة HTTP الصحيحة

```python
status.HTTP_200_OK          # نجاح عام
status.HTTP_201_CREATED     # بعد إنشاء مورد جديد (POST)
status.HTTP_204_NO_CONTENT  # نجاح بدون محتوى مُرجَع (DELETE)
status.HTTP_404_NOT_FOUND   # المورد غير موجود
status.HTTP_422_UNPROCESSABLE_ENTITY  # بيانات غير صالحة (يتم تلقائيًا من Pydantic)
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `status` من مثال سابق.

> ⚠️ **خطأ شائع:** إرجاع `200 OK` لكل شيء، حتى عند إنشاء مورد جديد (الأصح `201`) أو فشل العثور عليه (الأصح `404` بدل `200` مع رسالة خطأ داخل الجسم). الالتزام بمعايير REST يجعل واجهتك متوقعة وسهلة الاستخدام لأي عميل يتعامل معها.

### لا تثق أبدًا ببيانات العميل دون تحقق (هذا ما تمنحه لك Pydantic تلقائيًا)

## 19.12 تمارين

### 🟢 مستوى مبتدئ

**تمرين 1:** أنشئ واجهة API بسيطة بها نقطة نهاية `GET /greet/{name}` تُرجع `{"message": "مرحبًا، {name}!"}`.

**تمرين 2:** أضف نقطة نهاية `GET /calculate` تستقبل معاملي استعلام `a` و `b` (أعداد صحيحة)، وتُرجع ناتج جمعهما.

<details>
<summary>✅ الحل النموذجي</summary>

In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/greet/{name}")
def greet(name: str) -> dict[str, str]:
    return {"message": f"مرحبًا، {name}!"}

@app.get("/calculate")
def calculate(a: int, b: int) -> dict[str, int]:
    return {"result": a + b}

</details>

### 🟡 مستوى متوسط

**تمرين 3:** أضف إلى مشروع «إدارة المهام» نقطة نهاية `GET /tasks/stats` تُرجع عدد المهام الكلي، عدد المنجزة، وعدد غير المنجزة.

**تمرين 4:** أضف حقل `due_date: date | None` إلى نموذج `Task`، مع التحقق المناسب.

### 🔴 تحدي

**تمرين 5:** أضف نظام «فئات» (Categories) للمهام: نقطة نهاية جديدة `GET /categories` و `POST /categories`، وربط كل مهمة بـ `category_id` اختياري، مع التحقق من وجود الفئة قبل قبول المهمة.

## 19.13 اختبر فهمك (Quiz)

<details>
<summary>❓ السؤال 1: ما الفرق بين Path Parameter و Query Parameter؟</summary>

Path Parameter جزء من مسار الـ URL نفسه ويُحدد هوية مورد معيّن (`/tasks/5`). Query Parameter يُضاف بعد `?` ويُستخدم لتصفية أو تعديل سلوك الطلب (`/tasks?done=true`).
</details>

<details>
<summary>❓ السؤال 2: لماذا نستخدم Pydantic `BaseModel` بدلًا من قواميس عادية لاستقبال بيانات الطلب؟</summary>

لأن Pydantic يوفر تحققًا تلقائيًا وصارمًا من صحة البيانات (الأنواع، الحقول الإلزامية، القيود)، ويرفض الطلبات غير الصالحة تلقائيًا برسائل خطأ واضحة، دون الحاجة لكتابة أي كود تحقق يدوي.
</details>

<details>
<summary>❓ السؤال 3: ما الفرق بين رمزي الحالة 404 و 422؟</summary>

404 تعني أن المورد المطلوب غير موجود إطلاقًا (مثل مهمة بمعرّف غير موجود). 422 تعني أن البيانات المُرسَلة في جسم الطلب غير صالحة من حيث الصيغة أو النوع (مثل إرسال نص بدل رقم، أو نسيان حقل إلزامي).
</details>

## 19.14 ملخص الفصل (Cheat Sheet)

In [ ]:
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel, Field

app = FastAPI()

class ItemCreate(BaseModel):
    name: str = Field(min_length=1)
    price: float = Field(gt=0)

@app.get("/items/{item_id}")
def get_item(item_id: int, q: str | None = None) -> dict:
    ...

@app.post("/items", status_code=status.HTTP_201_CREATED)
def create_item(item: ItemCreate) -> ItemCreate:
    ...

@app.patch("/items/{item_id}")
def update_item(item_id: int, item: ItemCreate) -> ItemCreate:
    ...

@app.delete("/items/{item_id}", status_code=status.HTTP_204_NO_CONTENT)
def delete_item(item_id: int) -> None:
    ...

# تشغيل:
# uv run uvicorn main:app --reload
# التوثيق: http://127.0.0.1:8000/docs

| المفهوم | الاستخدام |
|---|---|
| `@app.get/post/put/delete` | تعريف نقطة نهاية |
| Path Parameter | `{id}` في المسار |
| Query Parameter | معامل دالة غير موجود بالمسار |
| `BaseModel` | نموذج بيانات الطلب/الاستجابة |
| `HTTPException` | إرجاع خطأ HTTP مناسب |
| `/docs` | توثيق تفاعلي تلقائي |

## 19.15 ماذا بعد؟

لقد بنيت للتو واجهة API حقيقية وكاملة! في الفصل القادم، ننتقل لمسار مختلف تمامًا، لكنه بنفس الأهمية: **تحليل البيانات**، حيث سنتعلم مكتبات `pandas` و`numpy` و`matplotlib` لتحليل وتصور بيانات حقيقية، عبر مشروع تحليل مبيعات متكامل.

---

### الخطوة التالية

- [الفصل العشرون: مسار تحليل البيانات (pandas, numpy, matplotlib)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-20.ipynb)
- [فهرس كل الدفاتر](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/00-index.ipynb)

*الفصل التاسع عشر — من كتاب «بايثون 3.14: من الصفر إلى الاحتراف».*